In [ ]:
!pip install snowflake-connector-python pandas pyarrow gcsfs google-cloud-storage -q
print("✅ Libraries installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 111.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 46.0.5 which is incompatible.
pydrive2 1.21.3 requires pyOpenSSL<=24.2.1,>=19.1.0, but you have pyopenssl 25.3.0 which is inco

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("✅ GCP Authenticated!")

✅ GCP Authenticated!


In [ ]:
# ⚠️ Replace ALL values with yours
PROJECT_ID    = "nyc-tlc-pipeline"
SILVER_BUCKET = "nyc-tlc-silver-prasad"   # e.g. nyc-tlc-silver-prasad
SILVER_PATH   = f"gs://{SILVER_BUCKET}/yellow_taxi/2024/"

# Snowflake credentials
SF_ACCOUNT   = "OUFSWVG-OZ74799"  # ⚠️ your account identifier
SF_USER      = "PRASADPATIL0103"              # ⚠️ your Snowflake username
SF_PASSWORD  = "Applepie@01032202"             # ⚠️ your Snowflake password
SF_DATABASE  = "NYC_TLC"
SF_SCHEMA    = "STAGING"
SF_WAREHOUSE = "COMPUTE_WH"

print("✅ Config set!")

✅ Config set!


In [ ]:
import snowflake.connector

conn = snowflake.connector.connect(
    account   = SF_ACCOUNT,
    user      = SF_USER,
    password  = SF_PASSWORD,
    database  = SF_DATABASE,
    schema    = SF_SCHEMA,
    warehouse = SF_WAREHOUSE
)

cursor = conn.cursor()
cursor.execute("SELECT CURRENT_VERSION(), CURRENT_DATABASE(), CURRENT_SCHEMA()")
row = cursor.fetchone()
print(f"✅ Connected to Snowflake!")
print(f"   Version  : {row[0]}")
print(f"   Database : {row[1]}")
print(f"   Schema   : {row[2]}")

✅ Connected to Snowflake!
   Version  : 10.6.2
   Database : NYC_TLC
   Schema   : STAGING


In [ ]:
import os
from google.cloud import storage

os.makedirs("/content/silver", exist_ok=True)
client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(SILVER_BUCKET)

blobs = list(bucket.list_blobs(prefix="yellow_taxi/2024/"))
print(f"Found {len(blobs)} files in Silver bucket")

for blob in blobs:
    if blob.name.endswith(".parquet"):
        # Flatten folder structure for local storage
        filename = blob.name.replace("/", "_")
        local_path = f"/content/silver/{filename}"
        print(f"⬇️ Downloading {blob.name}...")
        blob.download_to_filename(local_path)
        size_mb = os.path.getsize(local_path) / (1024*1024)
        print(f"✅ {filename} — {size_mb:.1f} MB")

print("\n🎉 All Silver files downloaded!")

Found 24 files in Silver bucket
⬇️ Downloading yellow_taxi/2024/pickup_year=2002/pickup_month=12/part-00003-042e585c-e1e6-4794-84d2-5f8c8d391b2a.c000.snappy.parquet...
✅ yellow_taxi_2024_pickup_year=2002_pickup_month=12_part-00003-042e585c-e1e6-4794-84d2-5f8c8d391b2a.c000.snappy.parquet — 0.0 MB
⬇️ Downloading yellow_taxi/2024/pickup_year=2008/pickup_month=12/part-00001-042e585c-e1e6-4794-84d2-5f8c8d391b2a.c000.snappy.parquet...
✅ yellow_taxi_2024_pickup_year=2008_pickup_month=12_part-00001-042e585c-e1e6-4794-84d2-5f8c8d391b2a.c000.snappy.parquet — 0.0 MB
⬇️ Downloading yellow_taxi/2024/pickup_year=2009/pickup_month=1/part-00001-042e585c-e1e6-4794-84d2-5f8c8d391b2a.c000.snappy.parquet...
✅ yellow_taxi_2024_pickup_year=2009_pickup_month=1_part-00001-042e585c-e1e6-4794-84d2-5f8c8d391b2a.c000.snappy.parquet — 0.0 MB
⬇️ Downloading yellow_taxi/2024/pickup_year=2009/pickup_month=1/part-00003-042e585c-e1e6-4794-84d2-5f8c8d391b2a.c000.snappy.parquet...
✅ yellow_taxi_2024_pickup_year=2009_pick

In [ ]:
import pandas as pd
import glob

files = glob.glob("/content/silver/*.parquet")
print(f"Found {len(files)} parquet files")

df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

print(f"✅ Loaded into pandas!")
print(f"📊 Total rows   : {len(df):,}")
print(f"📋 Total columns: {len(df.columns)}")
print(f"\nData types:\n{df.dtypes}")

Found 24 parquet files
✅ Loaded into pandas!
📊 Total rows   : 5,774,598
📋 Total columns: 17

Data types:
VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                   int32
trip_distance                   float64
PULocationID                      int32
DOLocationID                      int32
payment_type                      int32
fare_amount                     float64
tip_amount                      float64
total_amount                    float64
trip_duration_min               float64
revenue_per_mile                float64
pickup_hour                       int32
pickup_day                        int32
is_weekend                         bool
tip_percentage                  float64
dtype: object


In [ ]:
import pandas as pd

# Rename columns
df = df.rename(columns={
    "VendorID"             : "VENDORID",
    "tpep_pickup_datetime" : "PICKUP_DATETIME",
    "tpep_dropoff_datetime": "DROPOFF_DATETIME",
    "passenger_count"      : "PASSENGER_COUNT",
    "trip_distance"        : "TRIP_DISTANCE",
    "PULocationID"         : "PULOCATIONID",
    "DOLocationID"         : "DOLOCATIONID",
    "payment_type"         : "PAYMENT_TYPE",
    "fare_amount"          : "FARE_AMOUNT",
    "tip_amount"           : "TIP_AMOUNT",
    "total_amount"         : "TOTAL_AMOUNT",
    "trip_duration_min"    : "TRIP_DURATION_MIN",
    "revenue_per_mile"     : "REVENUE_PER_MILE",
    "pickup_hour"          : "PICKUP_HOUR",
    "pickup_day"           : "PICKUP_DAY",
    "pickup_month"         : "PICKUP_MONTH",
    "pickup_year"          : "PICKUP_YEAR",
    "is_weekend"           : "IS_WEEKEND",
    "tip_percentage"       : "TIP_PERCENTAGE"
})

# Check what columns we actually have
print(f"Columns ({len(df.columns)}):", df.columns.tolist())

# Add missing partition columns if not present
if "PICKUP_YEAR" not in df.columns:
    df["PICKUP_YEAR"] = pd.to_datetime(df["PICKUP_DATETIME"]).dt.year
    print("✅ Added PICKUP_YEAR")

if "PICKUP_MONTH" not in df.columns:
    df["PICKUP_MONTH"] = pd.to_datetime(df["PICKUP_DATETIME"]).dt.month
    print("✅ Added PICKUP_MONTH")

# Convert datetimes to plain strings
df["PICKUP_DATETIME"]  = pd.to_datetime(df["PICKUP_DATETIME"]).dt.strftime("%Y-%m-%d %H:%M:%S")
df["DROPOFF_DATETIME"] = pd.to_datetime(df["DROPOFF_DATETIME"]).dt.strftime("%Y-%m-%d %H:%M:%S")

# Fix nullable types
df["PASSENGER_COUNT"] = df["PASSENGER_COUNT"].astype(str).replace("<NA>", "")
df["PAYMENT_TYPE"]    = df["PAYMENT_TYPE"].astype(str).replace("<NA>", "")
df["IS_WEEKEND"]      = df["IS_WEEKEND"].astype(str)

# Reorder columns to exactly match Snowflake table
df = df[[
    "VENDORID", "PICKUP_DATETIME", "DROPOFF_DATETIME",
    "PASSENGER_COUNT", "TRIP_DISTANCE", "PULOCATIONID", "DOLOCATIONID",
    "PAYMENT_TYPE", "FARE_AMOUNT", "TIP_AMOUNT", "TOTAL_AMOUNT",
    "TRIP_DURATION_MIN", "REVENUE_PER_MILE", "PICKUP_HOUR", "PICKUP_DAY",
    "PICKUP_MONTH", "PICKUP_YEAR", "IS_WEEKEND", "TIP_PERCENTAGE"
]]

print(f"\n✅ Final columns ({len(df.columns)}): {df.columns.tolist()}")
print(f"\nSample:\n{df.head(2)}")

Columns (17): ['VENDORID', 'PICKUP_DATETIME', 'DROPOFF_DATETIME', 'PASSENGER_COUNT', 'TRIP_DISTANCE', 'PULOCATIONID', 'DOLOCATIONID', 'PAYMENT_TYPE', 'FARE_AMOUNT', 'TIP_AMOUNT', 'TOTAL_AMOUNT', 'TRIP_DURATION_MIN', 'REVENUE_PER_MILE', 'PICKUP_HOUR', 'PICKUP_DAY', 'IS_WEEKEND', 'TIP_PERCENTAGE']
✅ Added PICKUP_YEAR
✅ Added PICKUP_MONTH

✅ Final columns (19): ['VENDORID', 'PICKUP_DATETIME', 'DROPOFF_DATETIME', 'PASSENGER_COUNT', 'TRIP_DISTANCE', 'PULOCATIONID', 'DOLOCATIONID', 'PAYMENT_TYPE', 'FARE_AMOUNT', 'TIP_AMOUNT', 'TOTAL_AMOUNT', 'TRIP_DURATION_MIN', 'REVENUE_PER_MILE', 'PICKUP_HOUR', 'PICKUP_DAY', 'PICKUP_MONTH', 'PICKUP_YEAR', 'IS_WEEKEND', 'TIP_PERCENTAGE']

Sample:
   VENDORID      PICKUP_DATETIME     DROPOFF_DATETIME PASSENGER_COUNT  \
0         2  2024-03-01 00:01:37  2024-03-01 00:23:02               1   
1         2  2024-01-31 23:59:53  2024-02-01 00:18:35               1   

   TRIP_DISTANCE  PULOCATIONID  DOLOCATIONID PAYMENT_TYPE  FARE_AMOUNT  \
0           9.46      

In [ ]:
import math

os.makedirs("/content/chunks", exist_ok=True)
chunk_size = 500_000
n_chunks   = math.ceil(len(df) / chunk_size)

print(f"Splitting {len(df):,} rows into {n_chunks} chunks...")

for i in range(n_chunks):
    chunk = df.iloc[i*chunk_size:(i+1)*chunk_size]
    path  = f"/content/chunks/chunk_{i+1}.csv"
    chunk.to_csv(path, index=False)
    size_mb = os.path.getsize(path) / (1024*1024)
    print(f"✅ chunk_{i+1}.csv — {len(chunk):,} rows — {size_mb:.1f} MB")

print(f"\n🎉 {n_chunks} chunks ready!")

Splitting 5,774,598 rows into 12 chunks...
✅ chunk_1.csv — 500,000 rows — 51.0 MB
✅ chunk_2.csv — 500,000 rows — 51.0 MB
✅ chunk_3.csv — 500,000 rows — 51.0 MB
✅ chunk_4.csv — 500,000 rows — 51.0 MB
✅ chunk_5.csv — 500,000 rows — 51.0 MB
✅ chunk_6.csv — 500,000 rows — 51.0 MB
✅ chunk_7.csv — 500,000 rows — 51.0 MB
✅ chunk_8.csv — 500,000 rows — 51.0 MB
✅ chunk_9.csv — 500,000 rows — 51.0 MB
✅ chunk_10.csv — 500,000 rows — 51.0 MB
✅ chunk_11.csv — 500,000 rows — 51.0 MB
✅ chunk_12.csv — 274,598 rows — 28.0 MB

🎉 12 chunks ready!


In [ ]:
cursor = conn.cursor()

# Truncate table for clean load
cursor.execute("TRUNCATE TABLE NYC_TLC.STAGING.RAW_TRIPS")
print("✅ Table truncated!")

# Create internal stage
cursor.execute("CREATE OR REPLACE STAGE NYC_TLC.STAGING.TRIP_STAGE")
print("✅ Stage created!")

# Upload each chunk to Snowflake stage
for i in range(1, n_chunks + 1):
    path = f"/content/chunks/chunk_{i}.csv"
    print(f"⬆️  Staging chunk_{i}.csv...")
    cursor.execute(f"PUT file://{path} @NYC_TLC.STAGING.TRIP_STAGE AUTO_COMPRESS=TRUE")
    print(f"✅ chunk_{i} staged!")

# Load all staged files in one COPY INTO
print("\n⬇️  Running COPY INTO RAW_TRIPS...")
cursor.execute("""
    COPY INTO NYC_TLC.STAGING.RAW_TRIPS
    FROM @NYC_TLC.STAGING.TRIP_STAGE
    FILE_FORMAT = (
        TYPE                         = 'CSV'
        FIELD_OPTIONALLY_ENCLOSED_BY = '"'
        SKIP_HEADER                  = 1
        NULL_IF                      = ('', 'None', 'nan')
        TIMESTAMP_FORMAT             = 'YYYY-MM-DD HH24:MI:SS'
    )
    PURGE = TRUE
""")

rows = cursor.fetchall()
print(f"\n✅ COPY INTO complete!")
for row in rows:
    print(f"   {row}")

✅ Table truncated!
✅ Stage created!
⬆️  Staging chunk_1.csv...
✅ chunk_1 staged!
⬆️  Staging chunk_2.csv...
✅ chunk_2 staged!
⬆️  Staging chunk_3.csv...
✅ chunk_3 staged!
⬆️  Staging chunk_4.csv...
✅ chunk_4 staged!
⬆️  Staging chunk_5.csv...
✅ chunk_5 staged!
⬆️  Staging chunk_6.csv...
✅ chunk_6 staged!
⬆️  Staging chunk_7.csv...
✅ chunk_7 staged!
⬆️  Staging chunk_8.csv...
✅ chunk_8 staged!
⬆️  Staging chunk_9.csv...
✅ chunk_9 staged!
⬆️  Staging chunk_10.csv...
✅ chunk_10 staged!
⬆️  Staging chunk_11.csv...
✅ chunk_11 staged!
⬆️  Staging chunk_12.csv...
✅ chunk_12 staged!

⬇️  Running COPY INTO RAW_TRIPS...

✅ COPY INTO complete!
   ('trip_stage/chunk_10.csv.gz', 'LOADED', 500000, 500000, 1, 0, None, None, None, None)
   ('trip_stage/chunk_11.csv.gz', 'LOADED', 500000, 500000, 1, 0, None, None, None, None)
   ('trip_stage/chunk_5.csv.gz', 'LOADED', 500000, 500000, 1, 0, None, None, None, None)
   ('trip_stage/chunk_1.csv.gz', 'LOADED', 500000, 500000, 1, 0, None, None, None, None)
 

In [ ]:
cursor.execute("SELECT COUNT(*) FROM NYC_TLC.STAGING.RAW_TRIPS")
count = cursor.fetchone()[0]
print(f"✅ Total rows in Snowflake: {count:,}")

cursor.execute("""
    SELECT
        COUNT(*)                      AS total_trips,
        ROUND(AVG(TRIP_DISTANCE), 2)  AS avg_distance_miles,
        ROUND(AVG(TOTAL_AMOUNT), 2)   AS avg_fare_usd,
        ROUND(AVG(TIP_PERCENTAGE), 2) AS avg_tip_pct,
        MIN(PICKUP_DATETIME)          AS earliest_trip,
        MAX(PICKUP_DATETIME)          AS latest_trip
    FROM NYC_TLC.STAGING.RAW_TRIPS
""")
stats = cursor.fetchone()
print(f"\n📊 Quick Stats:")
print(f"   Total trips  : {stats[0]:,}")
print(f"   Avg distance : {stats[1]} miles")
print(f"   Avg fare     : ${stats[2]}")
print(f"   Avg tip      : {stats[3]}%")
print(f"   Date range   : {stats[4]} → {stats[5]}")

✅ Total rows in Snowflake: 5,774,598

📊 Quick Stats:
   Total trips  : 5,774,598
   Avg distance : 3.26 miles
   Avg fare     : $27.26
   Avg tip      : 12.04%
   Date range   : 2002-12-31 22:59:39 → 2024-03-01 00:01:37


In [ ]:
cursor.close()
conn.close()
print("✅ Snowflake connection closed!")
print("🎉 Day 3 Complete — 5.7M rows live in Snowflake STAGING.RAW_TRIPS!")

✅ Snowflake connection closed!
🎉 Day 3 Complete — 5.7M rows live in Snowflake STAGING.RAW_TRIPS!
